In [1]:
import urllib.parse
import urllib.request
import requests
import xml.etree.ElementTree as ET
from io import BytesIO
from PyPDF2 import PdfReader
import os
import time
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
import ollama

/Users/tanmayshubhgarg/Documents/Projects/Camp2025/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = ollama.Client()

model_name = "llama3"

In [3]:
sentence_transformer_model =  SentenceTransformer('all-MiniLM-L6-v2')

dimension = 384 # the minilm model has an output dimension of 384
doc_index = faiss.IndexFlatL2(dimension)
chunk_index = faiss.IndexFlatL2(dimension)

In [4]:
# question = input("Enter your question: ")
question = "If the Roman Empire had never fallen, how might the development of Western legal traditions, specifically concerning the rights of individuals versus the state, have differed from the actual historical trajectory?"

In [5]:
# Option 1 - Using the question directly as the keyword.
keyword = question

num_articles = 100
encodingmethod = "utf-8"
errortype = "strict"

In [6]:
encoded_search_term = urllib.parse.quote(keyword, encoding=encodingmethod, errors=errortype)
url = f'http://export.arxiv.org/api/query?search_query=all:{encoded_search_term}&start=0&max_results={num_articles}'

print(f"Searching for '{keyword}' on arXiv...")
print(f"URL: {url}")

try:
    response = urllib.request.urlopen(url)
    try:
        url_read = response.read().decode("utf-8")
    except UnicodeDecodeError:
        response = urllib.request.urlopen(url)
        url_read = response.read().decode("utf-8", errors="ignore")
    
    parse_xml = ET.fromstring(url_read)
    print("Successfully retrieved search results!")
except Exception as e:
    print(f"Error retrieving data: {e}")
    raise

Searching for 'If the Roman Empire had never fallen, how might the development of Western legal traditions, specifically concerning the rights of individuals versus the state, have differed from the actual historical trajectory?' on arXiv...
URL: http://export.arxiv.org/api/query?search_query=all:If%20the%20Roman%20Empire%20had%20never%20fallen%2C%20how%20might%20the%20development%20of%20Western%20legal%20traditions%2C%20specifically%20concerning%20the%20rights%20of%20individuals%20versus%20the%20state%2C%20have%20differed%20from%20the%20actual%20historical%20trajectory%3F&start=0&max_results=100
Successfully retrieved search results!


In [ ]:
ns = {"ns": "http://www.w3.org/2005/Atom"}
entries = parse_xml.findall('ns:entry', ns)

articles_data = []
for entry in entries:
    link = entry.find('ns:link[@type="application/pdf"]', ns)
    if link is not None and "href" in link.attrib:
        pdf_url = link.attrib['href']
        
        title = entry.find('ns:title', ns)
        title_text = title.text.strip() if title is not None else "Unknown Title"
        
        authors = entry.findall('ns:author/ns:name', ns)
        author_names = [author.text for author in authors] if authors else ["Unknown Author"]
        
        published = entry.find('ns:published', ns)
        published_date = published.text[:10] if published is not None else "Unknown Date"
        
        summary = entry.find('ns:summary', ns)
        summary_text = summary.text.strip() if summary is not None else "No summary available"
        
        metadata = {
            'title': title_text,
            'authors': author_names,
            'published': published_date,
            'summary': summary_text
        }
            
        articles_data.append({
            'pdf_url': pdf_url,
            'metadata': metadata
        })

print(f"Found {len(articles_data)} articles with PDF links")
for i, article in enumerate(articles_data):
    print(f"{i+1}. {article['metadata']['title'][:80]}...")

Found 100 articles with PDF links
1. Islamic Law, Western European Law and the Roots of Middle East's Long
  Divergen...
2. Redefining Accountability: Navigating Legal Challenges of Participant
  Liabilit...
3. Automated Refugee Case Analysis: An NLP Pipeline for Supporting Legal
  Practiti...
4. Understanding the Impact of Physicians' Legal Considerations on XAI
  Systems...
5. Towards A Structured Overview of Use Cases for Natural Language
  Processing in ...
6. Global recessions as a cascade phenomenon with heterogenous, interacting
  agent...
7. On Preemption and Overdetermination in Formal Theories of Causality...
8. Privacy Perspectives and Practices of Chinese Smart Home Product Teams...
9. Certifying and removing disparate impact...
10. Can AI be Consentful?...
11. Computer Modeling of Personal Autonomy and Legal Equilibrium...
12. CaseGNN: Graph Neural Networks for Legal Case Retrieval with
  Text-Attributed G...
13. Large Language Models as Fiduciaries: A Case Study Toward Ro

: 

In [ ]:
for article in articles_data:
        response = requests.get(article['pdf_url'])
        response.raise_for_status()
        
        pdf_reader = PdfReader(BytesIO(response.content))
        text_content = []

        for page in pdf_reader.pages:
            text_content.append(page.extract_text())

        full_text = "\n".join(text_content)

        if full_text:
            embedding = sentence_transformer_model.encode(full_text, convert_to_tensor=True)
            doc_index.add(np.array([embedding.numpy()]))

/Users/tanmayshubhgarg/Documents/Projects/Camp2025/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [ ]:
# Option 2 - Taking out stopwords

In [ ]:
# Option 3 - Ask the LLM to generate good keywords from the question.